# LangGraph Agentic RAG

This notebook runs the real document QA pipeline through all four Self-RAG reflections. It loads indexed chunks, retrieves and reranks candidates, filters irrelevant passages, verifies answer support, and checks answer utility.

The graph prepares bounded conversation context, decides whether retrieval is needed, rewrites the query, and then runs the existing RAG query path:

```text
START -> context_manager -> retrieval_gate[Ret] -> query_rewriter -> retrieve -> grade_relevance[Rel] -> generate -> verify_support[Sup] -> verify_utility[Use] -> persist/abstain -> END
```

This notebook covers [Ret], [Rel], [Sup], and [Use]. It does not implement regeneration, retrieval retry, or citation repair.

In [8]:
from __future__ import annotations

import sys
from pathlib import Path


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'src').is_dir():
            return candidate
    raise RuntimeError('Run this notebook from within the project directory tree.')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

Project root: /Users/humengqing/Documents/Code/VSCode/doc-qa-agent


In [9]:
from langgraph.checkpoint.memory import InMemorySaver
from IPython.display import Markdown, display

from src.agent.context import ContextManager
from src.agent.graph import build_agent_graph, invoke_agent_graph
from src.agent.relevance import LLMRelevanceGrader
from src.agent.rewrite import LLMQueryRewriter
from src.agent.routes import LLMRetrievalGate
from src.agent.support import LLMSupportVerifier
from src.agent.utility import LLMUtilityVerifier
from src.core.config import Config
from src.core.logger import setup_logging
from src.pipeline.query_runtime import build_query_pipeline

## Build the real RAG pipeline

This cell uses the same online query-runtime construction path as `main.py`. It loads existing chunks and document embeddings from Chroma, then rebuilds only the in-memory BM25 index. It does not parse documents or re-embed document chunks.

In [10]:
config = Config()
setup_logging(config)
pipeline = build_query_pipeline(config)

checkpointer = InMemorySaver()
graph = build_agent_graph(
    pipeline,
    retrieval_gate=LLMRetrievalGate(pipeline.llm),
    context_manager=ContextManager(config),
    query_rewriter=LLMQueryRewriter(pipeline.llm),
    relevance_grader=LLMRelevanceGrader(pipeline.llm),
    support_verifier=LLMSupportVerifier(pipeline.llm),
    utility_verifier=LLMUtilityVerifier(pipeline.llm),
    checkpointer=checkpointer,
)

2026-08-24 18:35:32,411 | INFO | src | Logging configured with level INFO


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

2026-08-24 18:35:38,487 | INFO | src.retrieval.embedder | Configured local embedding model: BAAI/bge-large-en-v1.5
2026-08-24 18:35:38,562 | INFO | src.retrieval.vector_store | Loaded 306 indexed chunk(s) from collection doc_chunks
2026-08-24 18:35:38,583 | INFO | src.retrieval.bm25_retriever | Built BM25 index for 306 chunk(s)
2026-08-24 18:35:38,594 | INFO | src.retrieval.reranker | Configured scadsai reranker: Qwen/Qwen3-Reranker-4B


## Visualize the LangGraph workflow

`draw_mermaid()` exposes the compiled graph structure. LangSmith records executions of this graph when `LANGSMITH_TRACING=true` and `LANGSMITH_API_KEY` are configured in `.env`.

In [11]:
mermaid_graph = graph.get_graph().draw_mermaid()
display(Markdown(f'```mermaid\n{mermaid_graph}\n```'))

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	retrieval_gate(retrieval_gate)
	context_manager(context_manager)
	query_rewriter(query_rewriter)
	retrieve(retrieve)
	grade_relevance(grade_relevance)
	generate(generate)
	verify_support(verify_support)
	trim_answer(trim_answer)
	verify_utility(verify_utility)
	abstain(abstain)
	persist_turn(persist_turn)
	__end__([<p>__end__</p>]):::last
	__start__ --> context_manager;
	abstain --> persist_turn;
	context_manager --> retrieval_gate;
	generate --> verify_support;
	grade_relevance -.-> abstain;
	grade_relevance -.-> generate;
	query_rewriter --> retrieve;
	retrieval_gate -.-> abstain;
	retrieval_gate -. &nbsp;retrieve&nbsp; .-> query_rewriter;
	retrieve --> grade_relevance;
	trim_answer --> verify_utility;
	verify_support -.-> abstain;
	verify_support -. &nbsp;retry&nbsp; .-> query_rewriter;
	verify_support -. &nbsp;trim&nbsp; .-> trim_answer;
	verify_support -. &nbsp;verify&nbsp; .-> verify_utility;
	verify_utility -.-> abstain;
	verify_utility -. &nbsp;persist&nbsp; .-> persist_turn;
	persist_turn --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

## Run an interactive conversation

Run this cell to start a multi-turn conversation. Every question uses the same `thread_id`, so the Query Rewriter can resolve follow-up references. Enter `exit`, `quit`, or `:q` to stop. Each turn performs the complete Ret-Rel-Sup-Use workflow. The current `InMemorySaver` is not durable across Python processes.

In [12]:
def run_interactive_chat(default_thread_id: str = 'notebook-interactive') -> str:
    thread_id = input(f'Thread ID [{default_thread_id}]: ').strip() or default_thread_id
    print('Enter exit, quit, or :q to stop the conversation.')

    while True:
        try:
            question = input('\nUser: ').strip()
        except EOFError:
            print('\nConversation ended.')
            break
            
        if question.casefold() in {'exit', 'quit', ':q'}:
            print('Conversation ended.')
            break
        if not question:
            continue

        print(f'\nUser: {question}')

        try:
            response = invoke_agent_graph(graph, question, thread_id=thread_id)
        except Exception as error:
            print(f'Error: {error}')
            continue

        display(Markdown(f'**Assistant**: {response.answer}'))
        for source in response.sources:
            page = source.page if source.page is not None else 'unknown'
            section = source.section_title or 'unknown'
            print(f'- {source.chunk_id} | {source.source} | page {page} | {section}')

    return thread_id

thread_id = run_interactive_chat()

Enter exit, quit, or :q to stop the conversation.

User: What does the confusion matrix show about the model's misclassification patterns?


2026-08-24 18:36:30,601 | INFO | src.generation.llm | Generated answer with 267 character(s)
2026-08-24 18:36:32,760 | INFO | src.generation.llm | Generated answer with 208 character(s)
2026-08-24 18:36:33,328 | INFO | src.retrieval.hybrid_retriever | Fused 40 dense and 40 BM25 result(s) into 30 chunk(s)
2026-08-24 18:36:33,748 | INFO | src.retrieval.reranker | Reranked 30 candidate(s) through ScaDS.AI
2026-08-24 18:36:46,975 | INFO | src.generation.llm | Generated answer with 1738 character(s)
2026-08-24 18:36:59,436 | INFO | src.generation.llm | Generated answer with 1512 character(s)
2026-08-24 18:36:59,438 | INFO | src.generation.rag_pipeline | Answered question with 5 retrieved source(s)
2026-08-24 18:37:26,286 | INFO | src.generation.llm | Generated answer with 3290 character(s)
2026-08-24 18:37:29,896 | INFO | src.generation.llm | Generated answer with 484 character(s)


**Assistant**: The confusion matrix is used to analyze the model's predictions on a test set, displaying the correctness and misclassification of the model when classifying. The matrix is analyzed based on key metrics: True Positive (TP), True Negative (TN), False Positive (FP), and False Negative (FN). 

The model's misclassification patterns show that the number of misclassifications for the 'Good' label is significantly lower than that for the 'Bad' label. The points in category 0 (Bad) appear to be more spread out than those in category 1 (Good), and the clustering is relatively loose, suggesting that the model is not as stable in extracting features from the ‘Bad’ categories as it is in extracting features from the ‘Good’ categories. 

The Grad-CAM heat map is used to explore the model's concerns and possible causes when misclassification occurs. It is observed that the model is not robust enough to noise and is easily affected by external disturbances, leading to misclassification. In one case, a sample labeled ‘Bad’ is misclassified as ‘Good’ due to noise in the vertical direction, and the model's focus is mainly concentrated in these noise areas rather than the internal structure of the 3D printed part. In another case, a sample labeled as 'Good' is misclassified as 'Bad' because the model fails to properly focus on the relevant features of the 3D printed part, instead being distracted by irrelevant regions.

HuMengqing_chunk_049, HuMengqing_chunk_118, HuMengqing_chunk_119, HuMengqing_chunk_120

- HuMengqing_chunk_049 | HuMengqing.pdf | page 27 | 2.2.4.2 Confusion Matrix
- HuMengqing_chunk_118 | HuMengqing.pdf | page 59 | 4.3.1 t-SNE
- HuMengqing_chunk_119 | HuMengqing.pdf | page 61 | 4.3.2 Grad-CAM
- HuMengqing_chunk_095 | HuMengqing.pdf | page 51 | 3.5 Performance Evaluation
- HuMengqing_chunk_120 | HuMengqing.pdf | page 61 | 4.3.2 Grad-CAM

User: "What does the Grad-CAM analysis reveal about why those misclassifications happen?


2026-08-24 18:37:37,940 | INFO | src.generation.llm | Generated answer with 294 character(s)
2026-08-24 18:37:40,624 | INFO | src.generation.llm | Generated answer with 355 character(s)
2026-08-24 18:37:40,909 | INFO | src.retrieval.hybrid_retriever | Fused 40 dense and 40 BM25 result(s) into 30 chunk(s)
2026-08-24 18:37:41,090 | INFO | src.retrieval.reranker | Reranked 30 candidate(s) through ScaDS.AI
2026-08-24 18:37:54,874 | INFO | src.generation.llm | Generated answer with 1811 character(s)
2026-08-24 18:38:02,616 | INFO | src.generation.llm | Generated answer with 1032 character(s)
2026-08-24 18:38:02,617 | INFO | src.generation.rag_pipeline | Answered question with 5 retrieved source(s)
2026-08-24 18:38:22,235 | INFO | src.generation.llm | Generated answer with 2525 character(s)
2026-08-24 18:38:24,894 | INFO | src.generation.llm | Generated answer with 351 character(s)


**Assistant**: The Grad-CAM analysis is used to explore the causes of misclassification in the 3D printed part model. The analysis reveals that the model is not robust enough to noise and is easily affected by external disturbances, leading to misclassification. In one case, the model misclassifies a sample labeled 'Bad' as 'Good' due to noise in the vertical direction, which interferes with the model's judgment. In another case, the model misclassifies a sample labeled 'Good' as 'Bad' because it focuses on the edges and upper air portion of the 3D printed part, rather than the critical structural areas. Additionally, the model's performance can be affected by preprocessing techniques like histogram matching, which can introduce noise or interference in certain cases. The model achieves an overall accuracy of 94%, but these misclassification cases demonstrate areas where the model's performance could be further improved.

Supported by chunk IDs: HuMengqing_chunk_119, HuMengqing_chunk_120, HuMengqing_chunk_121, HuMengqing_chunk_122.

- HuMengqing_chunk_119 | HuMengqing.pdf | page 61 | 4.3.2 Grad-CAM
- HuMengqing_chunk_120 | HuMengqing.pdf | page 61 | 4.3.2 Grad-CAM
- HuMengqing_chunk_121 | HuMengqing.pdf | page 61 | 4.3.2 Grad-CAM
- HuMengqing_chunk_122 | HuMengqing.pdf | page 61 | 4.3.2 Grad-CAM
- HuMengqing_chunk_115 | HuMengqing.pdf | page 58 | 4.3 Visualization Analysis
Conversation ended.


## Inspect checkpointed state

The checkpoint stores the state after the latest graph execution, including retrieval, relevance, support, and utility decisions, the original and rewritten query, and bounded conversation history. Inspect `snapshot.values` to review the latest turn.

In [13]:
graph_config = {'configurable': {'thread_id': thread_id}}
snapshot = graph.get_state(graph_config)
snapshot.values

{'question': '"What does the Grad-CAM analysis reveal about why those misclassifications happen?',
 'retrieval_action': 'retrieve',
 'retrieval_confidence': 0.8,
 'retrieval_reason': 'The request is asking for an explanation of a specific analysis technique, Grad-CAM, and its insights into model misclassifications, which could plausibly be found in documents related to machine learning or model interpretability.',
 'conversation_history': [{'role': 'user',
   'content': "What does the confusion matrix show about the model's misclassification patterns?"},
  {'role': 'assistant',
   'content': "The confusion matrix is used to analyze the model's predictions on a test set, displaying the correctness and misclassification of the model when classifying. The matrix is analyzed based on key metrics: True Positive (TP), True Negative (TN), False Positive (FP), and False Negative (FN). \n\nThe model's misclassification patterns show that the number of misclassifications for the 'Good' label is 

## Check LangSmith configuration safely

This cell checks whether tracing is configured without printing the API key. When enabled, open the `doc-qa-agent` project in the LangSmith website to inspect the retrieval gate decision and selected graph branch.

In [14]:
import os


langsmith_status = {
    'tracing_enabled': os.getenv('LANGSMITH_TRACING', '').lower() == 'true',
    'api_key_configured': bool(os.getenv('LANGSMITH_API_KEY')),
    'project': os.getenv('LANGSMITH_PROJECT', 'default'),
}
langsmith_status

{'tracing_enabled': True,
 'api_key_configured': True,
 'project': 'doc-qa-agent'}

## Runtime requirements

Build the index first with `python -m scripts.build_index` whenever source documents, parsing, chunking, filtering, or the embedding model changes. The online notebook requires `SCADS_API_KEY` for reranking and generation. It needs `MINERU_API_TOKEN` only when building the index. Configure `LANGSMITH_TRACING=true`, `LANGSMITH_API_KEY`, and `LANGSMITH_PROJECT=doc-qa-agent` to view traces in LangSmith. Do not run indexing against sensitive documents unless uploading their content to MinerU and ScaDS.AI is permitted.